# Lesson 01 - Bevezetés az AI ügynökökbe

Üdvözöllek az **AI Agents for Beginners** tanfolyam első leckéjében!

Egy **AI ügynök** olyan program, amely egy nagy nyelvi modellt (LLM) használ érvelőmotorjaként, és képes *műveleteket* végrehajtani a valós világban — API-k hívása, adatbázisok lekérdezése vagy kód futtatása — egy felhasználó nevében egy cél elérése érdekében.

Ebben a jegyzetfüzetben megépíted az első ügynöködet: egy **Utazási Ügynököt**, amely nyaralási célpontokat ajánl. Út közben megtanulod, hogyan:

1. Csatlakozz az Azure AI Foundry Agent Service-hez a **Microsoft Agent Framework** segítségével.
2. Adj az ügynöknek egy **eszközt** — egy sima Python függvényt, amelyet meghívhat.
3. Futtasd az ügynököt és vizsgáld meg a válaszát.
4. Tokenenként streameld az ügynök válaszát.


## Beállítás

A jegyzetfüzet futtatása előtt győződj meg arról, hogy rendelkezel:

1. **Egy Azure AI Foundry projekttel**, amelyben egy chat modell telepítve van (pl. `gpt-4o-mini`).
2. **Bejelentkezve az Azure CLI-vel** — futtasd az `az login` parancsot a terminálodban.
3. **Beállítva a szükséges környezeti változókat:**
   - `AZURE_AI_PROJECT_ENDPOINT` — az Azure AI Foundry projekted végpontja.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — a telepített modell neve.

Az alábbi cella telepíti a szükséges Python csomagokat.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Az első ügynök létrehozása

Egy ügynöknek két dologra van szüksége:

- **Utasítások**, amelyek megmondják neki, *ki ő* és *hogyan viselkedjen* (egy rendszerparancs).
- **Eszközök** — Python függvények, amelyeket `@tool` dekorátorral láttak el, és amelyeket az ügynök hívhat információk lekérése vagy műveletek végrehajtása céljából.

Az alábbiakban definiálunk egy egyszerű eszközt, amely népszerű nyaralási helyszínek listáját adja vissza. Az ügynök ezt az eszközt fogja használni, amikor egy felhasználó utazási ajánlásokat kér.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Válaszok

A még interaktívabb élmény érdekében **streamelheted** az ügynök válaszát. Teljes válasz várása helyett az ügynök az elkészült szövegrészleteket adja vissza. Ez különösen hasznos chat felületeken, ahol valós időben szeretnéd megjeleníteni a kimenetet.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Összefoglalás

Ebben a leckében megtanultad, hogyan:

- **Készíts szolgáltatót**, amely az Azure AI Foundry Agent Service-hez csatlakozik az `AzureAIProjectAgentProvider` segítségével.
- **Határozz meg egy eszközt** az `@tool` dekorátor használatával, hogy az ügynök hívhassa a Python függvényeidet.
- **Futtasd az ügynököt** egy felhasználói üzenettel, és írasd ki a válaszát.
- **Folyamatosan továbbítsd a válaszokat** valós idejű kimenet érdekében.

A következő leckében mélyebben megvizsgáljuk az ügynöki keretrendszereket, és megtanuljuk, hogyan adjunk az ügynököknek még hatékonyabb eszközöket és többlépéses érvelési képességeket.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
